<a href="https://colab.research.google.com/github/Hemamalini-L/Smart-Chennai-Public-Transport-Bottleneck-Analyzer/blob/main/Public_Transport_Bottleneck_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile

zip_path = "static_mtc_gtfs_data.zip"

with zipfile.ZipFile(zip_path, 'r') as z:


    trips = pd.read_csv(z.open('trips.txt'))
    stops = pd.read_csv(z.open('stops.txt'))
    stop_times = pd.read_csv(z.open('stop_times.txt'))
    calendar = pd.read_csv(z.open('calendar.txt'))

In [ ]:
import zipfile

zip_path = "static_mtc_gtfs_data.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    print(z.namelist())

In [ ]:
import zipfile

with zipfile.ZipFile("static_mtc_gtfs_data.zip", "r") as z:
    z.extractall("mtc_data")

print("Extraction completed")

In [ ]:
import os

for root, dirs, files in os.walk("mtc_data"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import zipfile

inner_zip = "mtc_data/static_mtc_gtfs_data/chennai.bus.gtfs.zip"

with zipfile.ZipFile(inner_zip, "r") as z:
    print(z.namelist())

In [ ]:
import pandas as pd
import zipfile

inner_zip = "mtc_data/static_mtc_gtfs_data/chennai.bus.gtfs.zip"

with zipfile.ZipFile(inner_zip, "r") as z:

    routes = pd.read_csv(z.open("chennai.bus.gtfs/routes.txt"))
    trips = pd.read_csv(z.open("chennai.bus.gtfs/trips.txt"))
    stops = pd.read_csv(z.open("chennai.bus.gtfs/stops.txt"))
    stop_times = pd.read_csv(z.open("chennai.bus.gtfs/stop_times.txt"))
    calendar = pd.read_csv(z.open("chennai.bus.gtfs/calendar.txt"))
    agency = pd.read_csv(z.open("chennai.bus.gtfs/agency.txt"))

print("Files Loaded Successfully!")

In [ ]:
print("Routes Shape :", routes.shape)
print("Trips Shape :", trips.shape)
print("Stops Shape :", stops.shape)
print("Stop Times Shape :", stop_times.shape)
print("Calendar Shape :", calendar.shape)

In [ ]:
print("Routes Columns")
print(routes.columns)

print("\nTrips Columns")
print(trips.columns)

print("\nStops Columns")
print(stops.columns)

print("\nStop Times Columns")
print(stop_times.columns)

In [ ]:
routes.head()
trips.head()
stops.head()
stop_times.head()


In [ ]:
print("Routes Missing Values")
print(routes.isnull().sum())

print("\nTrips Missing Values")
print(trips.isnull().sum())

print("\nStops Missing Values")
print(stops.isnull().sum())

print("\nStop Times Missing Values")
print(stop_times.isnull().sum())

In [ ]:
routes.drop_duplicates(inplace=True)
trips.drop_duplicates(inplace=True)
stops.drop_duplicates(inplace=True)
stop_times.drop_duplicates(inplace=True)

print("Duplicates Removed")

In [ ]:
stop_times["arrival_time"] = pd.to_datetime(
    stop_times["arrival_time"],
    format="%H:%M:%S",
    errors="coerce"
)

stop_times["departure_time"] = pd.to_datetime(
    stop_times["departure_time"],
    format="%H:%M:%S",
    errors="coerce"
)

In [ ]:
stop_times["hour"] = stop_times["arrival_time"].dt.hour

In [ ]:
stop_times[[
    "arrival_time",
    "hour"
]].head()

In [ ]:
stop_times["peak_hour"] = np.where(
    ((stop_times["hour"] >= 7) & (stop_times["hour"] <= 10))
    |
    ((stop_times["hour"] >= 17) & (stop_times["hour"] <= 20)),
    1,
    0
)

In [ ]:
stop_times["peak_hour"].value_counts()

In [ ]:
np.random.seed(42)

stop_times["delay_minutes"] = np.random.randint(
    0,
    20,
    len(stop_times)
)

In [ ]:
stop_times["delay_minutes"].head()

In [ ]:
stop_times["passenger_count"] = np.random.randint(
    10,
    80,
    len(stop_times)
)

In [ ]:
vehicle_capacity = 60

stop_times["occupancy_rate"] = (
    stop_times["passenger_count"] /
    vehicle_capacity
)

In [ ]:
stop_times[[
    "passenger_count",
    "occupancy_rate"
]].head()

In [ ]:
stop_times.to_csv(
    "clean_transport_data.csv",
    index=False
)

print("Dataset Saved Successfully")

**KPI CREATION**

In [ ]:
total_routes = routes["route_id"].nunique()

print("Total Routes:", total_routes)

In [ ]:
avg_delay = stop_times["delay_minutes"].mean()

print("Average Delay:", round(avg_delay,2), "minutes")

In [ ]:
passenger_volume = stop_times["passenger_count"].sum()

print("Passenger Volume:", passenger_volume)

In [ ]:
avg_occupancy = stop_times["occupancy_rate"].mean()

print("Average Occupancy Rate:", round(avg_occupancy,2))

In [ ]:
hourly_congestion = stop_times.groupby(
    "hour"
)["occupancy_rate"].mean()

hourly_congestion.sort_values(
    ascending=False
).head(5)

In [ ]:
peak_hours = hourly_congestion.sort_values(
    ascending=False
).head(5)

print(peak_hours)

In [ ]:
on_time_arrival = (
    stop_times["delay_minutes"] <= 5
).mean()*100

print(
    "On-Time Arrival:",
    round(on_time_arrival,2),
    "%"
)

In [ ]:
route_trip = trips[
    ["trip_id","route_id"]
]

analysis_df = stop_times.merge(
    route_trip,
    on="trip_id"
)

In [ ]:
high_risk_routes = analysis_df.groupby(
    "route_id"
)["delay_minutes"].mean()

In [ ]:
high_risk_routes = (
    high_risk_routes
    .sort_values(ascending=False)
    .head(10)
)

print(high_risk_routes)

In [ ]:
route_utilization = trips.groupby(
    "route_id"
).size().reset_index(
    name="trip_count"
)

In [ ]:
avg_route_utilization = (
    route_utilization["trip_count"]
    .mean()
)

print(
    "Average Route Utilization:",
    round(avg_route_utilization,2)
)

In [ ]:
kpi_summary = pd.DataFrame({

    "KPI":[
        "Total Routes",
        "Average Delay",
        "Passenger Volume",
        "Occupancy Rate",
        "On-Time Arrival %"
    ],

    "Value":[
        total_routes,
        round(avg_delay,2),
        passenger_volume,
        round(avg_occupancy,2),
        round(on_time_arrival,2)
    ]

})

kpi_summary

In [ ]:
kpi_summary.to_csv(
    "transport_kpi_summary.csv",
    index=False
)

print("KPI Summary Saved")

**BOTTLENECK INVESTIGATION**

In [ ]:
analysis_df = stop_times.merge(
    trips[['trip_id','route_id']],
    on='trip_id',
    how='left'
)

print(analysis_df.shape)
analysis_df.head()

In [ ]:
route_delay = analysis_df.groupby(
    'route_id'
)['delay_minutes'].mean().reset_index()

route_delay = route_delay.sort_values(
    'delay_minutes',
    ascending=False
)

route_delay.head(10)

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data=route_delay.head(10),
    x='route_id',
    y='delay_minutes'
)

plt.title("Top 10 Delayed Routes")
plt.xticks(rotation=90)

plt.show()

In [ ]:
stop_volume = analysis_df.groupby(
    'stop_id'
)['passenger_count'].sum().reset_index()

stop_volume = stop_volume.sort_values(
    'passenger_count',
    ascending=False
)

In [ ]:
stop_volume.head(10)

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data=stop_volume.head(10),
    x='stop_id',
    y='passenger_count'
)

plt.title("Top 10 Congested Stops")
plt.xticks(rotation=90)

plt.show()

In [ ]:
hourly_congestion = analysis_df.groupby(
    'hour'
)['occupancy_rate'].mean().reset_index()

In [ ]:
plt.figure(figsize=(12,6))

sns.lineplot(
    data=hourly_congestion,
    x='hour',
    y='occupancy_rate',
    marker='o'
)

plt.title("Hourly Congestion Trend")
plt.xlabel("Hour")
plt.ylabel("Occupancy Rate")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    analysis_df['delay_minutes'],
    bins=20,
    kde=True
)

plt.title("Delay Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    analysis_df['occupancy_rate'],
    bins=20,
    kde=True
)

plt.title("Occupancy Rate Distribution")

plt.show()

In [ ]:
route_utilization = trips.groupby(
    'route_id'
).size().reset_index(
    name='trip_count'
)

route_utilization = route_utilization.sort_values(
    'trip_count',
    ascending=False
)

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data=route_utilization.head(10),
    x='route_id',
    y='trip_count'
)

plt.title("Top 10 Utilized Routes")
plt.xticks(rotation=90)

plt.show()

In [ ]:
risk_df = analysis_df.groupby(
    'route_id'
).agg({

    'delay_minutes':'mean',
    'occupancy_rate':'mean'

}).reset_index()

In [ ]:
risk_df['risk_score'] = (
    risk_df['delay_minutes'] *
    risk_df['occupancy_rate']
)

In [ ]:
risk_df = risk_df.sort_values(
    'risk_score',
    ascending=False
)

risk_df.head(10)

In [ ]:
corr = analysis_df[[
    'delay_minutes',
    'passenger_count',
    'occupancy_rate',
    'hour'
]].corr()

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")

plt.show()

**WEATHER DATA**

In [ ]:
import requests
import pandas as pd

# Your OpenWeather API Key
API_KEY = "b773dc3ad118ed7993f7ef66223b77f4"

city = "Chennai"

url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

response = requests.get(url)

weather = response.json()

print(weather)

In [ ]:
weather_df = pd.DataFrame({
    "temperature": [weather["main"]["temp"]],
    "humidity": [weather["main"]["humidity"]],
    "wind_speed": [weather["wind"]["speed"]],
    "weather_condition": [weather["weather"][0]["description"]]
})

weather_df

In [ ]:
analysis_df["temperature"] = weather_df.loc[0, "temperature"]
analysis_df["humidity"] = weather_df.loc[0, "humidity"]
analysis_df["wind_speed"] = weather_df.loc[0, "wind_speed"]

In [ ]:
analysis_df[[
    "temperature",
    "humidity",
    "wind_speed"
]].head()

In [ ]:
print("Temperature:", weather_df.loc[0, "temperature"], "°C")
print("Humidity:", weather_df.loc[0, "humidity"], "%")
print("Wind Speed:", weather_df.loc[0, "wind_speed"], "m/s")
print("Condition:", weather_df.loc[0, "weather_condition"])

In [ ]:
weather_summary = pd.DataFrame({
    "Metric": ["Temperature", "Humidity", "Wind Speed"],
    "Value": [
        weather_df.loc[0, "temperature"],
        weather_df.loc[0, "humidity"],
        weather_df.loc[0, "wind_speed"]
    ]
})

weather_summary

In [ ]:
analysis_df.to_csv(
    "transport_analysis_dataset.csv",
    index=False
)

route_delay.to_csv(
    "route_delay_analysis.csv",
    index=False
)

route_utilization.to_csv(
    "route_utilization.csv",
    index=False
)

stop_volume.to_csv(
    "stop_passenger_volume.csv",
    index=False
)

kpi_summary.to_csv(
    "kpi_summary.csv",
    index=False
)

print("All files saved successfully")

In [ ]:
weather_history = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=365),
    "temperature": np.random.randint(25,40,365),
    "humidity": np.random.randint(50,95,365),
    "rainfall": np.random.uniform(0,20,365)
})

In [ ]:
weather_history.to_csv(
    "weather_dataset.csv",
    index=False
)

print("Weather dataset saved successfully")

In [ ]:
weather_history.head()

In [ ]:
avg_temp = weather_history["temperature"].mean()

print("Average Temperature:", round(avg_temp,2))

In [ ]:
avg_humidity = weather_history["humidity"].mean()

print("Average Humidity:", round(avg_humidity,2))

In [ ]:
avg_rainfall = weather_history["rainfall"].mean()

print("Average Rainfall:", round(avg_rainfall,2))

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    weather_history["date"],
    weather_history["temperature"]
)

plt.title("Temperature Trend")
plt.xlabel("Date")
plt.ylabel("Temperature")

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    weather_history["date"],
    weather_history["rainfall"]
)

plt.title("Rainfall Trend")
plt.xlabel("Date")
plt.ylabel("Rainfall")

plt.show()

In [ ]:
weather_corr = weather_history[
    ["temperature","humidity","rainfall"]
].corr()

weather_corr

In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    weather_corr,
    annot=True,
    cmap="coolwarm"
)

plt.title("Weather Correlation")

plt.show()

In [ ]:
weather_history.to_csv(
    "weather_dataset.csv",
    index=False
)

In [ ]:
print(weather_history.shape)
print(analysis_df.shape)
print(kpi_summary.shape)

In [ ]:

print("total_routes =", total_routes)
print("avg_delay =", avg_delay)
print("passenger_volume =", passenger_volume)
print("avg_occupancy =", avg_occupancy)
print("on_time_arrival =", on_time_arrival)

In [ ]:
dashboard_kpis = pd.DataFrame({
    "Metric": [
        "Total Routes",
        "Average Delay",
        "Passenger Volume",
        "Occupancy Rate",
        "On-Time Arrival %"
    ],
    "Value": [
        total_routes,
        round(avg_delay, 2),
        passenger_volume,
        round(avg_occupancy, 2),
        round(on_time_arrival, 2)
    ]
})

dashboard_kpis

In [ ]:
dashboard_kpis.to_csv(
    "dashboard_kpis.csv",
    index=False
)

print("dashboard_kpis.csv saved")

In [ ]:
import os

for file in os.listdir():
    print(file)

In [ ]:
from google.colab import files

files.download("dashboard_kpis.csv")
files.download("route_delay_analysis.csv")
files.download("route_utilization.csv")
files.download("stop_passenger_volume.csv")
files.download("transport_analysis_dataset.csv")
files.download("weather_dataset.csv")

In [ ]:
dashboard_data = analysis_df.sample(
    n=200000,
    random_state=42
)

dashboard_data.to_csv(
    "dashboard_data.csv",
    index=False
)

print(dashboard_data.shape)

In [ ]:
from google.colab import files
files.download("dashboard_data.csv")

In [ ]:
np.random.randint(0,20)

In [ ]:
analysis_df["delay_minutes"] = (
    analysis_df["peak_hour"] * 8
    + np.random.randint(0,5,len(analysis_df))
)

In [ ]:
analysis_df["passenger_count"] = np.where(
    analysis_df["peak_hour"] == 1,
    np.random.randint(40,80,len(analysis_df)),
    np.random.randint(10,40,len(analysis_df))
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [ ]:
y = analysis_df["delay_minutes"]

In [ ]:
X = analysis_df[
    [
        "hour",
        "occupancy_rate",
        "temperature",
        "humidity"
    ]
]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Features
X = analysis_df[
    ["hour", "occupancy_rate", "temperature", "humidity"]
]

# Target
y = analysis_df["delay_minutes"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

In [ ]:
import pickle

pickle.dump(
    model,
    open("delay_model.pkl", "wb")
)

In [ ]:
import os

os.listdir()

In [ ]:
from google.colab import files

files.download('/content/delay_model.pkl')

In [ ]:
import pickle

model = pickle.load(
    open("delay_model.pkl", "rb")
)

In [ ]:
analysis_df.to_csv(
    "final_analysis_dataset.csv",
    index=False
)

In [ ]:
files.download("final_analysis_dataset.csv")

In [ ]:
print(X.columns.tolist())

In [ ]:
import pandas as pd

print("dashboard_kpis")
print(pd.read_csv("dashboard_kpis.csv").head())

print("\nroute_delay_analysis")
print(pd.read_csv("route_delay_analysis.csv").head())

print("\nroute_utilization")
print(pd.read_csv("route_utilization.csv").head())

print("\nstop_passenger_volume")
print(pd.read_csv("stop_passenger_volume.csv").head())

print("\nweather_dataset")
print(pd.read_csv("weather_dataset.csv").head())

In [ ]:
import pandas as pd

df = pd.read_csv("dashboard_data.csv")

sample_df = df.sample(n=10000, random_state=42)

sample_df.to_csv("dashboard_data_sample.csv", index=False)

print(sample_df.shape)

In [ ]:
from google.colab import files

files.download("dashboard_data_sample.csv")

In [ ]:
import os

size_mb = os.path.getsize("dashboard_data.csv") / (1024 * 1024)

print(f"File Size: {size_mb:.2f} MB")

In [ ]:
import pandas as pd

df = pd.read_csv("dashboard_data_sample.csv")

print(df.columns.tolist())
print(df.shape)
df.head()